# TPO Ingeniería de Datos II — Exploración Multi-DB

Este notebook conecta directamente a las **4 bases de datos** del stack y permite:
- Ver los datos almacenados en cada una
- Insertar datos manualmente y verificar la propagación cross-DB
- Observar cómo se complementan bases relacionales y no relacionales

| Base | Puerto | Qué guarda |
|---|---|---|
| MongoDB 7 | 27017 | Usuarios + Propiedades |
| PostgreSQL 17 | 5432 | Reservas + Pagos |
| Cassandra 5 | 9042 | Reseñas |
| Neo4j 5 | 7687 | Grafo de relaciones |

> **Requisito:** tener los contenedores Docker corriendo (`docker compose up`)

## 0. Dependencias e imports

In [1]:
# Si algún driver no está instalado, descomentá la línea correspondiente
# !pip install pymongo psycopg2-binary cassandra-driver neo4j pandas tabulate

from pymongo import MongoClient
import psycopg2, psycopg2.extras
from cassandra.cluster import Cluster
from neo4j import GraphDatabase
import pandas as pd
import uuid, json
from datetime import date, datetime

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_columns', 20)
print('Imports OK')

Imports OK


## 1. Conexión a las 4 bases

In [2]:
# ── MongoDB ──────────────────────────────────────────────────────────────────
mongo_client = MongoClient('mongodb://localhost:27017/')
mongo_db     = mongo_client['airbnb_tpo']
print('MongoDB   :', mongo_client.server_info()['version'])

# ── PostgreSQL ────────────────────────────────────────────────────────────────
pg_conn = psycopg2.connect(
    host='localhost', port=5432,
    dbname='airbnb_tpo', user='airbnb', password='airbnb'
)
pg_conn.autocommit = False
pg_cur = pg_conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
pg_cur.execute('SELECT version()')
print('PostgreSQL:', pg_cur.fetchone()['version'].split(',')[0])

# ── Cassandra ─────────────────────────────────────────────────────────────────
cass_cluster = Cluster(['localhost'])
cass_session = cass_cluster.connect('airbnb_tpo')
row = cass_session.execute('SELECT release_version FROM system.local').one()
print('Cassandra :', row.release_version)

# ── Neo4j ─────────────────────────────────────────────────────────────────────
neo4j_driver  = GraphDatabase.driver('bolt://localhost:7687', auth=('neo4j', 'airbnb_tpo_2026'))
with neo4j_driver.session() as s:
    v = s.run('RETURN "Neo4j conectado" AS msg').single()['msg']
    print('Neo4j     :', v)

MongoDB   : 7.0.34
PostgreSQL: PostgreSQL 17.10 (Debian 17.10-1.pgdg13+1) on aarch64-unknown-linux-gnu
Cassandra : 5.0.8
Neo4j     : Neo4j conectado


---
## 2. MongoDB — Usuarios y Propiedades

MongoDB almacena documentos con estructura flexible. Las propiedades tienen un campo `ubicacion.coords` de tipo GeoJSON con un índice `2dsphere` para búsquedas geoespaciales.

In [3]:
# ── Usuarios ──────────────────────────────────────────────────────────────────
usuarios = list(mongo_db.usuarios.find({}, {'_id': 0}))
df_usuarios = pd.DataFrame(usuarios)[['id','nombre','email','tipo','promedio_rating','created_at']]
print(f'Total usuarios: {len(df_usuarios)}')
df_usuarios

Total usuarios: 11


,id,nombre,email,tipo,promedio_rating,created_at
0,11111111-0001-0001-0001-000000000001,Ana Gómez,ana.host@mail.com,anfitrion,4.57,2025-01-01T00:00:00.000Z
1,11111111-0002-0001-0001-000000000002,Carlos Pérez,carlos.host@mail.com,anfitrion,0.00,2025-01-01T00:00:00.000Z
2,22222222-0001-0001-0001-000000000001,Lucía QA,lucia@mail.com,huesped,0.00,2025-01-01T00:00:00.000Z
3,22222222-0002-0001-0001-000000000002,Martín Silva,martin@mail.com,huesped,0.00,2025-01-01T00:00:00.000Z
4,0291695a-0ef3-4f50-8d73-c0edd8ab386f,Test Huésped,test.huesped@qa.com,huesped,0.00,2026-05-20T22:01:39.421Z
5,09d22da3-015f-488d-b52b-82b511df34c1,Test Anfitrión,test.anfitrion@qa.com,anfitrion,0.00,2026-05-20T22:01:39.535Z
6,f4b210fd-4a57-46c5-8b38-cf0ce3640e02,Test Huésped,test.huesped@qa.com,huesped,0.00,2026-05-20T22:02:16.394Z
7,47601f4e-f744-48ec-aee0-d1e7adaa1195,Test Anfitrión,test.anfitrion@qa.com,anfitrion,0.00,2026-05-20T22:02:16.413Z
8,3a74174f-103f-4237-a79c-6f9d71b95e4e,Test Huésped,test.huesped@qa.com,huesped,0.00,2026-05-20T22:03:42.909Z
9,9c308c94-5bde-4a2a-bb0c-70320994854a,Test Anfitrión,test.anfitrion@qa.com,anfitrion,0.00,2026-05-20T22:03:42.939Z


In [4]:
# ── Propiedades ───────────────────────────────────────────────────────────────
propiedades = list(mongo_db.propiedades.find({'estado': {'$ne': 'eliminada'}}, {'_id': 0}))
rows = []
for p in propiedades:
    rows.append({
        'id':            p['id'],
        'titulo':        p['titulo'],
        'tipo':          p['tipo'],
        'ciudad':        p['ubicacion']['ciudad'],
        'precio_noche':  p['precio_noche'],
        'capacidad':     p['cantidad_huespedes'],
        'rating':        p['promedio_rating'],
        'coords':        p['ubicacion']['coords']['coordinates'],
    })
df_props = pd.DataFrame(rows)
print(f'Total propiedades activas: {len(df_props)}')
df_props

Total propiedades activas: 5


,id,titulo,tipo,ciudad,precio_noche,capacidad,rating,coords
0,33333333-0001-0001-0001-000000000001,Depto luminoso en Palermo,departamento,Buenos Aires,60000,3,5.0,"[-58.421, -34.589]"
1,33333333-0002-0001-0001-000000000002,Casa con patio en Córdoba,casa,Córdoba,80000,5,4.0,"[-64.188, -31.42]"
2,33333333-0003-0001-0001-000000000003,Loft céntrico en Rosario,loft,Rosario,50000,2,0.0,"[-60.64, -32.95]"
3,c66995ee-ef32-423c-a297-eb7ff40d9bac,Pepe,departamento,Buenos Aires,50000,2,0.0,"[-58.4, -34.6]"
4,e33ca01e-f20c-4f84-8bec-363321e94aed,Cabaña en Bariloche,cabana,Bariloche,120000,4,5.0,"[-71.31, -41.133]"


In [5]:
# ── Índices de MongoDB ────────────────────────────────────────────────────────
print('Índices en colección propiedades:')
for name, idx in mongo_db.propiedades.index_information().items():
    print(f'  {name:30s} → {idx["key"]}')

print('\nÍndices en colección usuarios:')
for name, idx in mongo_db.usuarios.index_information().items():
    print(f'  {name:30s} → {idx["key"]}')

Índices en colección propiedades:
  _id_                           → [('_id', 1)]
  ubicacion.coords_2dsphere      → [('ubicacion.coords', '2dsphere')]
  id_1                           → [('id', 1)]

Índices en colección usuarios:
  _id_                           → [('_id', 1)]
  id_1                           → [('id', 1)]


In [6]:
# ── Búsqueda geoespacial (índice 2dsphere) ────────────────────────────────────
# Propiedades dentro de 100 km de Buenos Aires
BA_LNG, BA_LAT = -58.421, -34.589
RADIO_M = 100_000  # 100 km en metros

geo_result = list(mongo_db.propiedades.find(
    {
        'estado': {'$ne': 'eliminada'},
        'ubicacion.coords': {
            '$near': {
                '$geometry': {'type': 'Point', 'coordinates': [BA_LNG, BA_LAT]},
                '$maxDistance': RADIO_M
            }
        }
    },
    {'_id': 0, 'titulo': 1, 'ubicacion.ciudad': 1, 'precio_noche': 1}
))

print(f'Propiedades dentro de 100 km de Buenos Aires: {len(geo_result)}')
for p in geo_result:
    print(f'  • {p["titulo"]} — {p["ubicacion"]["ciudad"]} — ${p["precio_noche"]:,}')

Propiedades dentro de 100 km de Buenos Aires: 2
  • Depto luminoso en Palermo — Buenos Aires — $60,000
  • Pepe — Buenos Aires — $50,000


---
## 3. PostgreSQL — Reservas y Pagos

PostgreSQL garantiza **ACID** en las operaciones de reserva. Las tablas `reservas` y `pagos` tienen integridad referencial (`FOREIGN KEY`). Los UUIDs de `huesped_id`, `anfitrion_id` y `propiedad_id` referencian documentos en MongoDB.

In [7]:
# ── Schema de las tablas ──────────────────────────────────────────────────────
pg_cur.execute("""
    SELECT column_name, data_type, is_nullable, column_default
    FROM information_schema.columns
    WHERE table_name = 'reservas'
    ORDER BY ordinal_position
""")
print('Tabla: reservas')
display(pd.DataFrame(pg_cur.fetchall()))

pg_cur.execute("""
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_name = 'pagos'
    ORDER BY ordinal_position
""")
print('\nTabla: pagos')
display(pd.DataFrame(pg_cur.fetchall()))

Tabla: reservas


,column_name,data_type,is_nullable,column_default
0,id,uuid,NO,None
1,huesped_id,uuid,NO,None
2,anfitrion_id,uuid,NO,None
3,propiedad_id,uuid,NO,None
4,fecha_inicio,date,NO,None
5,fecha_fin,date,NO,None
6,cantidad_huespedes,integer,NO,None
7,estado,character varying,NO,None
8,created_at,timestamp without time zone,NO,now()



Tabla: pagos


,column_name,data_type,is_nullable
0,id,uuid,NO
1,reserva_id,uuid,NO
2,monto,numeric,NO
3,metodo,character varying,NO
4,estado,character varying,NO
5,created_at,timestamp without time zone,NO


In [8]:
# ── Todas las reservas con su pago (JOIN) ─────────────────────────────────────
pg_cur.execute("""
    SELECT
        r.id,
        r.huesped_id,
        r.propiedad_id,
        r.fecha_inicio,
        r.fecha_fin,
        r.cantidad_huespedes,
        r.estado AS estado_reserva,
        p.monto,
        p.metodo,
        p.estado AS estado_pago,
        r.created_at
    FROM reservas r
    JOIN pagos p ON p.reserva_id = r.id
    ORDER BY r.created_at DESC
""")
reservas = pg_cur.fetchall()
print(f'Total reservas: {len(reservas)}')
df_reservas = pd.DataFrame(reservas)
df_reservas

Total reservas: 11


,id,huesped_id,propiedad_id,fecha_inicio,fecha_fin,cantidad_huespedes,estado_reserva,monto,metodo,estado_pago,created_at
0,6ea1810a-3eb3-49b9-9b9c-12e90644f07d,ca4f9543-4c83-484c-b42c-58988a8e387c,e33ca01e-f20c-4f84-8bec-363321e94aed,2027-03-10,2027-03-15,3,completada,600000.00,transferencia,completado,2026-05-20 22:13:50.274452
1,29ddca4c-aac1-4331-a214-a32b3ed6bca3,22222222-0002-0001-0001-000000000002,33333333-0002-0001-0001-000000000002,2026-09-01,2026-09-03,2,completada,160000.00,efectivo,completado,2026-05-20 22:03:43.145446
2,dce7af4d-99be-43b5-a41b-90309279f3ff,22222222-0001-0001-0001-000000000001,33333333-0001-0001-0001-000000000001,2026-08-04,2026-08-07,2,confirmada,180000.00,transferencia,pendiente,2026-05-20 22:03:43.104045
3,6b5f877b-0442-41fa-90ec-e67a20cec2c0,22222222-0001-0001-0001-000000000001,33333333-0001-0001-0001-000000000001,2026-08-04,2026-08-07,2,cancelada,180000.00,tarjeta,pendiente,2026-05-20 22:03:43.050104
4,ff5a979f-5b55-4054-b37a-7e727645f448,22222222-0001-0001-0001-000000000001,33333333-0001-0001-0001-000000000001,2026-08-01,2026-08-04,2,completada,180000.00,tarjeta,completado,2026-05-20 22:03:43.021737
5,2c795d74-9f27-48c7-8f1a-6abe85255d00,22222222-0002-0001-0001-000000000002,33333333-0002-0001-0001-000000000002,2026-09-01,2026-09-03,2,completada,160000.00,efectivo,completado,2026-05-20 22:02:16.598924
6,8453d3e0-a29e-449d-8de3-082069dc0ab3,22222222-0001-0001-0001-000000000001,33333333-0001-0001-0001-000000000001,2026-08-01,2026-08-04,2,completada,180000.00,tarjeta,completado,2026-05-20 22:02:16.496342
7,55efaedc-547d-4542-b33e-68443518a131,22222222-0002-0001-0001-000000000002,33333333-0002-0001-0001-000000000002,2026-09-01,2026-09-03,2,completada,160000.00,efectivo,completado,2026-05-20 22:01:39.883069
8,febc10ac-3190-4809-be7d-47e85552fe63,22222222-0001-0001-0001-000000000001,33333333-0001-0001-0001-000000000001,2026-08-04,2026-08-07,2,cancelada,180000.00,transferencia,pendiente,2026-05-20 22:01:39.816468
9,bb2f6af5-7148-4105-bae8-9d4366e8c84f,22222222-0001-0001-0001-000000000001,33333333-0001-0001-0001-000000000001,2026-08-04,2026-08-07,2,cancelada,180000.00,tarjeta,pendiente,2026-05-20 22:01:39.777965


In [9]:
# ── Estadísticas por estado ───────────────────────────────────────────────────
pg_cur.execute("""
    SELECT
        r.estado,
        COUNT(*)           AS cantidad,
        SUM(p.monto)       AS facturado_total,
        AVG(p.monto)       AS ticket_promedio,
        AVG(r.fecha_fin - r.fecha_inicio) AS noches_promedio
    FROM reservas r
    JOIN pagos p ON p.reserva_id = r.id
    GROUP BY r.estado
    ORDER BY cantidad DESC
""")
display(pd.DataFrame(pg_cur.fetchall()))

,estado,cantidad,facturado_total,ticket_promedio,noches_promedio
0,completada,7,1620000.00,231428.571428571429,2.8571428571428571
1,cancelada,3,540000.00,180000.000000000000,3.0000000000000000
2,confirmada,1,180000.00,180000.000000000000,3.0000000000000000


In [10]:
# ── Constraints y claves foráneas ─────────────────────────────────────────────
pg_cur.execute("""
    SELECT
        tc.table_name,
        tc.constraint_name,
        tc.constraint_type,
        kcu.column_name,
        ccu.table_name  AS foreign_table,
        ccu.column_name AS foreign_column
    FROM information_schema.table_constraints tc
    JOIN information_schema.key_column_usage kcu
        ON tc.constraint_name = kcu.constraint_name
    LEFT JOIN information_schema.constraint_column_usage ccu
        ON ccu.constraint_name = tc.constraint_name
    WHERE tc.table_name IN ('reservas', 'pagos')
    ORDER BY tc.table_name, tc.constraint_type
""")
print('Constraints y FK en reservas/pagos:')
display(pd.DataFrame(pg_cur.fetchall()))

Constraints y FK en reservas/pagos:


,table_name,constraint_name,constraint_type,column_name,foreign_table,foreign_column
0,pagos,pagos_reserva_id_fkey,FOREIGN KEY,reserva_id,reservas,id
1,pagos,pagos_pkey,PRIMARY KEY,id,pagos,id
2,pagos,pagos_reserva_id_key,UNIQUE,reserva_id,pagos,reserva_id
3,reservas,reservas_pkey,PRIMARY KEY,id,reservas,id


---
## 4. Cassandra — Reseñas

Cassandra usa el patrón **"una tabla por query"**: los mismos datos se guardan desnormalizados en 3 tablas para optimizar cada tipo de consulta sin necesidad de JOINs.

| Tabla | Partition Key | Caso de uso |
|---|---|---|
| `resenias_by_propiedad` | `propiedad_id` | Ver reseñas de un alojamiento |
| `resenias_by_anfitrion` | `anfitrion_id` | Ver historial de un anfitrión |
| `resenias_by_reserva` | `reserva_id` | Verificar si una reserva ya fue reseñada |

In [11]:
# ── Schema de las tablas Cassandra ────────────────────────────────────────────
for tabla in ['resenias_by_propiedad', 'resenias_by_anfitrion', 'resenias_by_reserva']:
    rows = cass_session.execute(
        "SELECT column_name, type, kind FROM system_schema.columns "
        "WHERE keyspace_name = 'airbnb_tpo' AND table_name = %s",
        [tabla]
    )
    print(f'\n── {tabla} ──')
    display(pd.DataFrame(list(rows)))


── resenias_by_propiedad ──


,column_name,type,kind
0,anfitrion_id,text,regular
1,calificacion,int,regular
2,comentario,text,regular
3,created_at,timestamp,clustering
4,huesped_id,text,regular
5,id,text,clustering
6,propiedad_id,text,partition_key
7,reserva_id,text,regular



── resenias_by_anfitrion ──


,column_name,type,kind
0,anfitrion_id,text,partition_key
1,calificacion,int,regular
2,comentario,text,regular
3,created_at,timestamp,clustering
4,huesped_id,text,regular
5,id,text,clustering
6,propiedad_id,text,regular
7,reserva_id,text,regular



── resenias_by_reserva ──


,column_name,type,kind
0,anfitrion_id,text,regular
1,calificacion,int,regular
2,comentario,text,regular
3,created_at,timestamp,regular
4,huesped_id,text,regular
5,id,text,regular
6,propiedad_id,text,regular
7,reserva_id,text,partition_key


In [12]:
# ── Todas las reseñas ─────────────────────────────────────────────────────────
rows = list(cass_session.execute('SELECT * FROM resenias_by_reserva'))
print(f'Total reseñas: {len(rows)}')
if rows:
    display(pd.DataFrame(rows))
else:
    print('(sin reseñas todavía — creá una reserva completada primero)')

Total reseñas: 7


,reserva_id,anfitrion_id,calificacion,comentario,created_at,huesped_id,id,propiedad_id
0,2c795d74-9f27-48c7-8f1a-6abe85255d00,11111111-0001-0001-0001-000000000001,4,Muy buena,2026-05-20 22:02:16.624,22222222-0002-0001-0001-000000000002,bccb2f00-9ace-49d3-bcda-ff9c5e8ab83d,33333333-0002-0001-0001-000000000002
1,8453d3e0-a29e-449d-8de3-082069dc0ab3,11111111-0001-0001-0001-000000000001,5,Excelente,2026-05-20 22:02:16.581,22222222-0001-0001-0001-000000000001,123cd455-66f5-48b4-956d-ea9dd851ca18,33333333-0001-0001-0001-000000000001
2,ff5a979f-5b55-4054-b37a-7e727645f448,11111111-0001-0001-0001-000000000001,5,Excelente,2026-05-20 22:03:43.129,22222222-0001-0001-0001-000000000001,4880ba87-7ca7-4454-a76d-b9c6da49664a,33333333-0001-0001-0001-000000000001
3,7c2514c1-8a78-418c-b37f-28ba04f97d5f,11111111-0001-0001-0001-000000000001,5,Excelente,2026-05-20 22:01:39.843,22222222-0001-0001-0001-000000000001,45ceff0a-05c4-4292-94d9-1edf7209030e,33333333-0001-0001-0001-000000000001
4,55efaedc-547d-4542-b33e-68443518a131,11111111-0001-0001-0001-000000000001,4,Muy buena,2026-05-20 22:01:39.911,22222222-0002-0001-0001-000000000002,45cdb55d-1084-426d-a77c-45f68a9ed67a,33333333-0002-0001-0001-000000000002
5,6ea1810a-3eb3-49b9-9b9c-12e90644f07d,11111111-0001-0001-0001-000000000001,5,Impresionante vista a los lagos. Volvería sin dudarlo.,2026-05-20 22:13:50.913,ca4f9543-4c83-484c-b42c-58988a8e387c,a81dd502-0484-492d-8b23-40c10f395def,e33ca01e-f20c-4f84-8bec-363321e94aed
6,29ddca4c-aac1-4331-a214-a32b3ed6bca3,11111111-0001-0001-0001-000000000001,4,Muy buena,2026-05-20 22:03:43.169,22222222-0002-0001-0001-000000000002,b4b869e4-3c04-4acc-a867-abcd10a077b4,33333333-0002-0001-0001-000000000002


In [13]:
# ── Reseñas por propiedad (query optimizada por partition key) ────────────────
PROP1 = '33333333-0001-0001-0001-000000000001'

rows = list(cass_session.execute(
    'SELECT * FROM resenias_by_propiedad WHERE propiedad_id = %s',
    [PROP1]
))
print(f'Reseñas de la propiedad {PROP1}: {len(rows)}')
if rows:
    display(pd.DataFrame(rows)[['id','calificacion','comentario','huesped_id','created_at']])

Reseñas de la propiedad 33333333-0001-0001-0001-000000000001: 3


,id,calificacion,comentario,huesped_id,created_at
0,4880ba87-7ca7-4454-a76d-b9c6da49664a,5,Excelente,22222222-0001-0001-0001-000000000001,2026-05-20 22:03:43.129
1,123cd455-66f5-48b4-956d-ea9dd851ca18,5,Excelente,22222222-0001-0001-0001-000000000001,2026-05-20 22:02:16.581
2,45ceff0a-05c4-4292-94d9-1edf7209030e,5,Excelente,22222222-0001-0001-0001-000000000001,2026-05-20 22:01:39.843


In [14]:
# ── Promedio de calificaciones por propiedad (desde Cassandra) ────────────────
import collections

todas = list(cass_session.execute('SELECT propiedad_id, calificacion FROM resenias_by_reserva'))
if todas:
    grupos = collections.defaultdict(list)
    for r in todas:
        grupos[r.propiedad_id].append(r.calificacion)
    
    resumen = [{
        'propiedad_id': pid,
        'cantidad_resenias': len(califs),
        'promedio': round(sum(califs)/len(califs), 2)
    } for pid, califs in grupos.items()]
    display(pd.DataFrame(resumen))
else:
    print('Sin reseñas para resumir')

,propiedad_id,cantidad_resenias,promedio
0,33333333-0002-0001-0001-000000000002,3,4.0
1,33333333-0001-0001-0001-000000000001,3,5.0
2,e33ca01e-f20c-4f84-8bec-363321e94aed,1,5.0


---
## 5. Neo4j — Grafo de Relaciones

Neo4j modela las conexiones entre entidades como un grafo:

```
(:Usuario)-[:RESERVO]->(:Propiedad)
(:Usuario)-[:ANFITRIO]->(:Propiedad)
```

Esto permite hacer **collaborative filtering** en Cypher: *"Huéspedes que reservaron lo mismo que tú también reservaron X"*.

In [15]:
# ── Todos los nodos ───────────────────────────────────────────────────────────
with neo4j_driver.session() as s:
    resultado = s.run('MATCH (n) RETURN labels(n)[0] AS tipo, count(*) AS cantidad')
    print('Nodos en el grafo:')
    display(pd.DataFrame([r.data() for r in resultado]))

Nodos en el grafo:


,tipo,cantidad
0,Usuario,11
1,Propiedad,8


In [16]:
# ── Todas las relaciones ──────────────────────────────────────────────────────
with neo4j_driver.session() as s:
    resultado = s.run('MATCH ()-[r]->() RETURN type(r) AS relacion, count(*) AS cantidad')
    print('Relaciones en el grafo:')
    display(pd.DataFrame([r.data() for r in resultado]))

Relaciones en el grafo:


,relacion,cantidad
0,ANFITRIO,8
1,RESERVO,3


In [17]:
# ── Detalle: quién reservó qué ────────────────────────────────────────────────
with neo4j_driver.session() as s:
    resultado = s.run("""
        MATCH (u:Usuario)-[:RESERVO]->(p:Propiedad)
        RETURN u.nombre AS huesped, u.id AS huesped_id,
               p.titulo AS propiedad, p.ciudad AS ciudad
        ORDER BY huesped
    """)
    rows = [r.data() for r in resultado]
    if rows:
        display(pd.DataFrame(rows))
    else:
        print('Sin relaciones RESERVO todavía')

,huesped,huesped_id,propiedad,ciudad
0,Lucía QA,22222222-0001-0001-0001-000000000001,Depto luminoso en Palermo,Buenos Aires
1,Martín Silva,22222222-0002-0001-0001-000000000002,Casa con patio en Córdoba,Córdoba
2,Roberto Notebook,ca4f9543-4c83-484c-b42c-58988a8e387c,Cabaña en Bariloche,Bariloche


In [18]:
# ── Detalle: quién es anfitrión de qué ───────────────────────────────────────
with neo4j_driver.session() as s:
    resultado = s.run("""
        MATCH (u:Usuario)-[:ANFITRIO]->(p:Propiedad)
        RETURN u.nombre AS anfitrion, p.titulo AS propiedad, p.ciudad AS ciudad
        ORDER BY anfitrion
    """)
    display(pd.DataFrame([r.data() for r in resultado]))

,anfitrion,propiedad,ciudad
0,Ana Gómez,Cabaña en Bariloche,Bariloche
1,Ana Gómez,Prop QA Test,Mendoza
2,Ana Gómez,Prop QA Test,Mendoza
3,Ana Gómez,Prop QA Test,Mendoza
4,Ana Gómez,Pepe,Buenos Aires
5,Ana Gómez,Casa con patio en Córdoba,Córdoba
6,Ana Gómez,Depto luminoso en Palermo,Buenos Aires
7,Carlos Pérez,Loft céntrico en Rosario,Rosario


In [19]:
# ── Query de recomendaciones colaborativas para un usuario ────────────────────
GUEST1 = '22222222-0001-0001-0001-000000000001'

with neo4j_driver.session() as s:
    resultado = s.run("""
        MATCH (u:Usuario {id: $id})-[:RESERVO]->(p:Propiedad)
              <-[:RESERVO]-(otros:Usuario)-[:RESERVO]->(rec:Propiedad)
        WHERE NOT (u)-[:RESERVO]->(rec)
        RETURN rec.id AS propiedad_id, rec.titulo AS titulo,
               rec.ciudad AS ciudad, count(*) AS score
        ORDER BY score DESC
        LIMIT 5
    """, id=GUEST1)
    rows = [r.data() for r in resultado]

if rows:
    print(f'Recomendaciones colaborativas para {GUEST1}:')
    display(pd.DataFrame(rows))
else:
    print('Sin recomendaciones colaborativas (no hay otros usuarios con historial similar).')
    print('→ En ese caso la API usa fallback por promedio_rating desde MongoDB.')

Sin recomendaciones colaborativas (no hay otros usuarios con historial similar).
→ En ese caso la API usa fallback por promedio_rating desde MongoDB.


---
## 6. INSERT: Flujo completo cross-DB

Simulamos el ciclo de vida completo de una reserva e insertamos en todas las bases:

```
1. Crear usuario (huésped)         → MongoDB
2. Crear propiedad                 → MongoDB  +  Neo4j (nodo + relación ANFITRIO)
3. Crear reserva                   → PostgreSQL  +  Neo4j (relación RESERVO)
4. Finalizar reserva               → PostgreSQL (transacción: reserva + pago)
5. Crear reseña                    → Cassandra (3 tablas)  +  MongoDB (cache para rating)
6. Verificar en todas las bases    → cross-DB check
```

In [20]:
# IDs de referencia del seed (UUIDs fijos)
HOST1  = '11111111-0001-0001-0001-000000000001'
HOST2  = '11111111-0002-0001-0001-000000000002'
GUEST1 = '22222222-0001-0001-0001-000000000001'
GUEST2 = '22222222-0002-0001-0001-000000000002'
PROP1  = '33333333-0001-0001-0001-000000000001'
PROP2  = '33333333-0002-0001-0001-000000000002'
PROP3  = '33333333-0003-0001-0001-000000000003'
print('IDs de seed cargados')

IDs de seed cargados


In [21]:
# ── PASO 1: Nuevo huésped en MongoDB ─────────────────────────────────────────
nuevo_guest_id = str(uuid.uuid4())
nuevo_guest = {
    'id':             nuevo_guest_id,
    'nombre':         'Roberto Notebook',
    'email':          f'roberto_{nuevo_guest_id[:8]}@notebook.com',
    'telefono':       '9999-0000',
    'tipo':           'huesped',
    'bio':            'Usuario creado desde el notebook de exploración.',
    'promedio_rating': 0,
    'created_at':     datetime.utcnow().isoformat() + 'Z',
}
mongo_db.usuarios.insert_one(nuevo_guest)

# Sincronizar nodo en Neo4j
with neo4j_driver.session() as s:
    s.run(
        'MERGE (u:Usuario {id: $id}) SET u.nombre = $nombre, u.tipo = $tipo',
        id=nuevo_guest_id, nombre=nuevo_guest['nombre'], tipo='huesped'
    )

print(f'✓ Nuevo huésped creado en MongoDB + Neo4j: {nuevo_guest["nombre"]} ({nuevo_guest_id})')

✓ Nuevo huésped creado en MongoDB + Neo4j: Roberto Notebook (d27b3089-1fd1-4835-b144-7c6481625474)


/var/folders/ml/z35t_p1543j87f0g_ss6t9n40000gn/T/ipykernel_1260/2693602974.py:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at':     datetime.utcnow().isoformat() + 'Z',


In [22]:
# ── PASO 2: Nueva propiedad en MongoDB ───────────────────────────────────────
nueva_prop_id = str(uuid.uuid4())
nueva_prop = {
    'id':                nueva_prop_id,
    'anfitrion_id':      HOST1,
    'titulo':            'Cabaña en Bariloche',
    'tipo':              'cabana',
    'ubicacion': {
        'ciudad':     'Bariloche',
        'pais':       'Argentina',
        'direccion':  'Cerro Otto 500',
        'coords':     {'type': 'Point', 'coordinates': [-71.310, -41.133]},
    },
    'precio_noche':      120000,
    'descripcion':       'Vista a los lagos, tranquilidad total.',
    'cantidad_huespedes': 4,
    'servicios':         ['wifi', 'calefacción', 'parrilla', 'estacionamiento'],
    'estado':            'activa',
    'promedio_rating':   0,
    'imagen':            'https://images.unsplash.com/photo-1506905925346-21bda4d32df4?auto=format&fit=crop&w=900&q=80',
}
mongo_db.propiedades.insert_one(nueva_prop)

# Sincronizar en Neo4j (nodo Propiedad + relación ANFITRIO)
with neo4j_driver.session() as s:
    s.run("""
        MERGE (p:Propiedad {id: $id})
        SET p.titulo = $titulo, p.ciudad = $ciudad, p.tipo = $tipo
        MERGE (u:Usuario {id: $anfitrion_id})
        MERGE (u)-[:ANFITRIO]->(p)
    """, id=nueva_prop_id, titulo=nueva_prop['titulo'],
         ciudad='Bariloche', tipo='cabana', anfitrion_id=HOST1)

print(f'✓ Nueva propiedad creada en MongoDB + Neo4j: {nueva_prop["titulo"]} ({nueva_prop_id})')

✓ Nueva propiedad creada en MongoDB + Neo4j: Cabaña en Bariloche (e5da2179-9499-4268-9ea8-9d090646c111)


In [23]:
# ── PASO 3: Nueva reserva en PostgreSQL (con transacción) ────────────────────
reserva_id  = str(uuid.uuid4())
pago_id     = str(uuid.uuid4())
fecha_ini   = '2027-03-10'
fecha_fin   = '2027-03-15'   # 5 noches
noches      = 5
monto       = noches * nueva_prop['precio_noche']  # 5 × 120000 = 600000

try:
    pg_cur.execute("""
        INSERT INTO reservas (id, huesped_id, anfitrion_id, propiedad_id,
                              fecha_inicio, fecha_fin, cantidad_huespedes, estado)
        VALUES (%s, %s, %s, %s, %s, %s, %s, 'confirmada')
    """, (reserva_id, nuevo_guest_id, HOST1, nueva_prop_id,
           fecha_ini, fecha_fin, 3))

    pg_cur.execute("""
        INSERT INTO pagos (id, reserva_id, monto, metodo, estado)
        VALUES (%s, %s, %s, 'transferencia', 'pendiente')
    """, (pago_id, reserva_id, monto))

    pg_conn.commit()
    print(f'✓ Reserva creada en PostgreSQL (transacción COMMIT)')
    print(f'  reserva_id : {reserva_id}')
    print(f'  pago_id    : {pago_id}')
    print(f'  monto      : ${monto:,}')
except Exception as e:
    pg_conn.rollback()
    print(f'✗ Error (ROLLBACK): {e}')

# Sincronizar relación RESERVO en Neo4j
with neo4j_driver.session() as s:
    s.run("""
        MERGE (u:Usuario {id: $huesped_id})
        MERGE (p:Propiedad {id: $propiedad_id})
        MERGE (u)-[:RESERVO]->(p)
    """, huesped_id=nuevo_guest_id, propiedad_id=nueva_prop_id)
print('✓ Relación RESERVO registrada en Neo4j')

✓ Reserva creada en PostgreSQL (transacción COMMIT)
  reserva_id : 4dba4c48-1d89-4901-b91d-5e2afeb36749
  pago_id    : 50771f54-e578-40ef-9cc4-9c9c3bba05df
  monto      : $600,000


✓ Relación RESERVO registrada en Neo4j


In [24]:
# ── PASO 4: Finalizar reserva (transacción: reserva + pago atómico) ───────────
try:
    pg_cur.execute(
        "UPDATE reservas SET estado = 'completada' WHERE id = %s RETURNING id",
        (reserva_id,)
    )
    pg_cur.execute(
        "UPDATE pagos SET estado = 'completado' WHERE reserva_id = %s",
        (reserva_id,)
    )
    pg_conn.commit()
    print(f'✓ Reserva finalizada → estado: completada, pago: completado')
except Exception as e:
    pg_conn.rollback()
    print(f'✗ Error (ROLLBACK): {e}')

✓ Reserva finalizada → estado: completada, pago: completado


In [25]:
# ── PASO 5: Reseña en Cassandra (batch en 3 tablas) ───────────────────────────
resenia_id  = str(uuid.uuid4())
calificacion = 5
comentario   = 'Impresionante vista a los lagos. Volvería sin dudarlo.'
ts           = datetime.utcnow()

batch = """
BEGIN BATCH
  INSERT INTO resenias_by_propiedad
    (propiedad_id, created_at, id, reserva_id, huesped_id, anfitrion_id, calificacion, comentario)
    VALUES (%(pid)s, %(ts)s, %(rid)s, %(resid)s, %(hid)s, %(aid)s, %(cal)s, %(com)s);
  INSERT INTO resenias_by_anfitrion
    (anfitrion_id, created_at, id, reserva_id, propiedad_id, huesped_id, calificacion, comentario)
    VALUES (%(aid)s, %(ts)s, %(rid)s, %(resid)s, %(pid)s, %(hid)s, %(cal)s, %(com)s);
  INSERT INTO resenias_by_reserva
    (reserva_id, id, propiedad_id, anfitrion_id, huesped_id, calificacion, comentario, created_at)
    VALUES (%(resid)s, %(rid)s, %(pid)s, %(aid)s, %(hid)s, %(cal)s, %(com)s, %(ts)s);
APPLY BATCH;
"""

# Cassandra no soporta %(named)s — usamos execute con un dict y el driver
from cassandra.query import BatchStatement, SimpleStatement
from cassandra import ConsistencyLevel

batch_stmt = BatchStatement(consistency_level=ConsistencyLevel.ONE)

stmt1 = cass_session.prepare(
    'INSERT INTO resenias_by_propiedad '
    '(propiedad_id,created_at,id,reserva_id,huesped_id,anfitrion_id,calificacion,comentario) '
    'VALUES (?,?,?,?,?,?,?,?)'
)
stmt2 = cass_session.prepare(
    'INSERT INTO resenias_by_anfitrion '
    '(anfitrion_id,created_at,id,reserva_id,propiedad_id,huesped_id,calificacion,comentario) '
    'VALUES (?,?,?,?,?,?,?,?)'
)
stmt3 = cass_session.prepare(
    'INSERT INTO resenias_by_reserva '
    '(reserva_id,id,propiedad_id,anfitrion_id,huesped_id,calificacion,comentario,created_at) '
    'VALUES (?,?,?,?,?,?,?,?)'
)

batch_stmt.add(stmt1, (nueva_prop_id, ts, resenia_id, reserva_id, nuevo_guest_id, HOST1, calificacion, comentario))
batch_stmt.add(stmt2, (HOST1, ts, resenia_id, reserva_id, nueva_prop_id, nuevo_guest_id, calificacion, comentario))
batch_stmt.add(stmt3, (reserva_id, resenia_id, nueva_prop_id, HOST1, nuevo_guest_id, calificacion, comentario, ts))

cass_session.execute(batch_stmt)
print(f'✓ Reseña guardada en Cassandra (BATCH en 3 tablas)')
print(f'  resenia_id   : {resenia_id}')
print(f'  calificacion : {calificacion}/5')

✓ Reseña guardada en Cassandra (BATCH en 3 tablas)
  resenia_id   : 5197ddc8-4b1b-40fe-bea5-40496ac9efd7
  calificacion : 5/5


/var/folders/ml/z35t_p1543j87f0g_ss6t9n40000gn/T/ipykernel_1260/4043024474.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts           = datetime.utcnow()


In [26]:
# Cache en MongoDB (colección auxiliar para recalcular promedios)
mongo_db.resenias_cache.insert_one({
    'id':           resenia_id,
    'reserva_id':   reserva_id,
    'propiedad_id': nueva_prop_id,
    'anfitrion_id': HOST1,
    'huesped_id':   nuevo_guest_id,
    'calificacion': calificacion,
})

# Recalcular promedio de la propiedad
resenias_prop = list(mongo_db.resenias_cache.find({'propiedad_id': nueva_prop_id}))
nuevo_rating  = round(sum(r['calificacion'] for r in resenias_prop) / len(resenias_prop), 2)
mongo_db.propiedades.update_one({'id': nueva_prop_id}, {'$set': {'promedio_rating': nuevo_rating}})

# Recalcular promedio del anfitrión
resenias_host = list(mongo_db.resenias_cache.find({'anfitrion_id': HOST1}))
nuevo_rating_host = round(sum(r['calificacion'] for r in resenias_host) / len(resenias_host), 2)
mongo_db.usuarios.update_one({'id': HOST1}, {'$set': {'promedio_rating': nuevo_rating_host}})

print(f'✓ Ratings recalculados en MongoDB')
print(f'  Propiedad rating : {nuevo_rating}')
print(f'  Anfitrión rating : {nuevo_rating_host}')

✓ Ratings recalculados en MongoDB
  Propiedad rating : 5.0
  Anfitrión rating : 4.62


---
## 7. Verificación cross-DB: la reserva desde todas las perspectivas

Tomamos la reserva recién creada y mostramos cómo cada base aporta una pieza distinta de información.

In [27]:
print('═' * 60)
print(f'RESERVA: {reserva_id}')
print('═' * 60)

# ── PostgreSQL: datos transaccionales ────────────────────────────────────────
pg_cur.execute("""
    SELECT r.estado, r.fecha_inicio, r.fecha_fin, r.cantidad_huespedes,
           p.monto, p.metodo, p.estado AS estado_pago
    FROM reservas r JOIN pagos p ON p.reserva_id = r.id
    WHERE r.id = %s
""", (reserva_id,))
pg_data = pg_cur.fetchone()
print('\n[PostgreSQL] — transacción:')
for k, v in pg_data.items():
    print(f'  {k}: {v}')

# ── MongoDB: datos del huésped y la propiedad ─────────────────────────────────
huesped   = mongo_db.usuarios.find_one({'id': nuevo_guest_id},   {'_id': 0})
propiedad = mongo_db.propiedades.find_one({'id': nueva_prop_id}, {'_id': 0})
print('\n[MongoDB] — quién reservó y qué:')
print(f'  Huésped   : {huesped["nombre"]} ({huesped["tipo"]})')
print(f'  Propiedad : {propiedad["titulo"]} en {propiedad["ubicacion"]["ciudad"]}')
print(f'  Precio    : ${propiedad["precio_noche"]:,}/noche — rating: {propiedad["promedio_rating"]}')

# ── Cassandra: reseña asociada ────────────────────────────────────────────────
row = cass_session.execute(
    'SELECT * FROM resenias_by_reserva WHERE reserva_id = %s', [reserva_id]
).one()
print('\n[Cassandra] — reseña:')
if row:
    print(f'  Calificación : {row.calificacion}/5')
    print(f'  Comentario   : "{row.comentario}"')
    print(f'  Fecha        : {row.created_at}')
else:
    print('  (sin reseña)')

# ── Neo4j: relaciones del grafo ───────────────────────────────────────────────
with neo4j_driver.session() as s:
    rel = s.run("""
        MATCH (u:Usuario {id: $hid})-[r:RESERVO]->(p:Propiedad {id: $pid})
        RETURN type(r) AS relacion, u.nombre AS huesped, p.titulo AS propiedad
    """, hid=nuevo_guest_id, pid=nueva_prop_id).single()
print('\n[Neo4j] — relación en el grafo:')
if rel:
    print(f'  ({rel["huesped"]})-[:{rel["relacion"]}]->({rel["propiedad"]})')
else:
    print('  (sin relación)')

print('\n═' * 60)

════════════════════════════════════════════════════════════
RESERVA: 4dba4c48-1d89-4901-b91d-5e2afeb36749
════════════════════════════════════════════════════════════



[PostgreSQL] — transacción:
  estado: completada
  fecha_inicio: 2027-03-10
  fecha_fin: 2027-03-15
  cantidad_huespedes: 3
  monto: 600000.00
  metodo: transferencia
  estado_pago: completado

[MongoDB] — quién reservó y qué:
  Huésped   : Roberto Notebook (huesped)
  Propiedad : Cabaña en Bariloche en Bariloche
  Precio    : $120,000/noche — rating: 5.0

[Cassandra] — reseña:
  Calificación : 5/5
  Comentario   : "Impresionante vista a los lagos. Volvería sin dudarlo."
  Fecha        : 2026-05-20 22:15:52.058000



[Neo4j] — relación en el grafo:
  (Roberto Notebook)-[:RESERVO]->(Cabaña en Bariloche)

═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═


---
## 8. Demostración de separación de responsabilidades

Esta celda muestra por qué **no podría usarse una sola base** para todo.

In [28]:
# ── Caso 1: ACID en PostgreSQL — intento de reserva con fechas solapadas ──────
print('CASO 1: PostgreSQL — integridad de solapamiento de fechas')
print('-' * 50)

# Intentar insertar una reserva que se solapa con la existente (Mar 10-15)
try:
    pg_cur.execute("""
        SELECT id FROM reservas
        WHERE propiedad_id = %s
          AND estado = 'confirmada'
          AND fecha_inicio < %s
          AND fecha_fin > %s
        LIMIT 1
    """, (nueva_prop_id, '2027-03-13', '2027-03-12'))
    conflicto = pg_cur.fetchone()
    # Nuestra reserva es 'completada', así que no conflicto para nuevas confirmadas
    if conflicto:
        print(f'  ✗ Solapamiento detectado → reserva rechazada (ROLLBACK)')
    else:
        print(f'  ✓ Rango disponible (la reserva existente está completada, no bloquea)')
except Exception as e:
    print(f'  Error: {e}')

# Demostrar FK constraint: intentar insertar pago con reserva_id inválido
print('\nIntento de violar FK (pago con reserva inexistente):')
try:
    pg_cur.execute("""
        INSERT INTO pagos (id, reserva_id, monto, metodo, estado)
        VALUES (%s, %s, 100, 'tarjeta', 'pendiente')
    """, (str(uuid.uuid4()), str(uuid.uuid4())))
    pg_conn.commit()
    print('  ✗ No debería llegar aquí')
except Exception as e:
    pg_conn.rollback()
    print(f'  ✓ FK violación rechazada → {type(e).__name__}')

CASO 1: PostgreSQL — integridad de solapamiento de fechas
--------------------------------------------------
  ✓ Rango disponible (la reserva existente está completada, no bloquea)

Intento de violar FK (pago con reserva inexistente):
  ✓ FK violación rechazada → ForeignKeyViolation


In [29]:
# ── Caso 2: Cassandra — consulta eficiente por partition key ──────────────────
print('CASO 2: Cassandra — eficiencia de partition key vs ALLOW FILTERING')
print('-' * 50)

import time

# Query con partition key (O(1) en distribución)
start = time.perf_counter()
rows_pk = list(cass_session.execute(
    'SELECT * FROM resenias_by_propiedad WHERE propiedad_id = %s', [nueva_prop_id]
))
t_pk = (time.perf_counter() - start) * 1000
print(f'  Con partition key (propiedad_id): {len(rows_pk)} resultados en {t_pk:.1f} ms')

# Query completa (scan de tabla)
start = time.perf_counter()
rows_all = list(cass_session.execute('SELECT * FROM resenias_by_reserva'))
t_all = (time.perf_counter() - start) * 1000
print(f'  Sin partition key (scan total) : {len(rows_all)} resultados en {t_all:.1f} ms')
print(f'\n  → En producción con millones de filas, la diferencia sería de órdenes de magnitud.')

CASO 2: Cassandra — eficiencia de partition key vs ALLOW FILTERING
--------------------------------------------------


  Con partition key (propiedad_id): 1 resultados en 2.2 ms
  Sin partition key (scan total) : 8 resultados en 2.7 ms

  → En producción con millones de filas, la diferencia sería de órdenes de magnitud.


In [30]:
# ── Caso 3: MongoDB — flexibilidad de schema (campo nuevo sin migración) ───────
print('CASO 3: MongoDB — agregar campo nuevo sin migración de schema')
print('-' * 50)

# Agregar campo 'verificado' solo a los anfitriones (no afecta a los huéspedes)
result = mongo_db.usuarios.update_many(
    {'tipo': 'anfitrion'},
    {'$set': {'verificado': True, 'verificado_en': datetime.utcnow().isoformat()}}
)
print(f'  Documentos actualizados: {result.modified_count}')

# Verificar que los huéspedes NO tienen el campo (sin romper nada)
anfitriones = list(mongo_db.usuarios.find({'tipo': 'anfitrion'}, {'_id': 0, 'nombre': 1, 'verificado': 1}))
huespedes   = list(mongo_db.usuarios.find({'tipo': 'huesped'},   {'_id': 0, 'nombre': 1, 'verificado': 1}))

print('\n  Anfitriones:')
for u in anfitriones:
    print(f'    {u["nombre"]:20s} → verificado: {u.get("verificado", "(campo ausente)")}')
print('  Huéspedes:')
for u in huespedes[:2]:
    print(f'    {u["nombre"]:20s} → verificado: {u.get("verificado", "(campo ausente)")}')

# En SQL esto requeriría ALTER TABLE + default value + migración
print('\n  → En SQL esto requeriría ALTER TABLE (bloqueante en tablas grandes).')

CASO 3: MongoDB — agregar campo nuevo sin migración de schema
--------------------------------------------------


  Documentos actualizados: 5

  Anfitriones:
    Ana Gómez            → verificado: True
    Carlos Pérez         → verificado: True
    Test Anfitrión       → verificado: True
    Test Anfitrión       → verificado: True
    Test Anfitrión       → verificado: True
  Huéspedes:
    Lucía QA             → verificado: (campo ausente)
    Martín Silva         → verificado: (campo ausente)

  → En SQL esto requeriría ALTER TABLE (bloqueante en tablas grandes).


/var/folders/ml/z35t_p1543j87f0g_ss6t9n40000gn/T/ipykernel_1260/3775101195.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  {'$set': {'verificado': True, 'verificado_en': datetime.utcnow().isoformat()}}


In [31]:
# ── Caso 4: Neo4j — traversal de grafo (imposible en SQL sin CTEs recursivas) ──
print('CASO 4: Neo4j — encontrar usuarios "a 2 saltos" de una propiedad')
print('-' * 50)

with neo4j_driver.session() as s:
    # Todos los huéspedes que reservaron propiedades del mismo anfitrión que HOST1
    resultado = s.run("""
        MATCH (anfitrion:Usuario {id: $host})-[:ANFITRIO]->(p:Propiedad)
              <-[:RESERVO]-(huesped:Usuario)
        RETURN DISTINCT huesped.nombre AS huesped, p.titulo AS propiedad
        ORDER BY huesped
    """, host=HOST1)
    rows = [r.data() for r in resultado]

if rows:
    print(f'  Huéspedes que reservaron propiedades de {"Ana Gómez"} (HOST1):')
    for r in rows:
        print(f'    {r["huesped"]} → {r["propiedad"]}')
else:
    print('  (sin datos de traversal todavía — hacé más reservas)')

print('\n  → En SQL: requeriría JOIN de 3 tablas. En Neo4j: 1 query de traversal.')

CASO 4: Neo4j — encontrar usuarios "a 2 saltos" de una propiedad
--------------------------------------------------
  Huéspedes que reservaron propiedades de Ana Gómez (HOST1):
    Lucía QA → Depto luminoso en Palermo
    Martín Silva → Casa con patio en Córdoba
    Roberto Notebook → Cabaña en Bariloche

  → En SQL: requeriría JOIN de 3 tablas. En Neo4j: 1 query de traversal.


---
## 9. Estado final de todas las bases

In [32]:
# ── Resumen de conteos en cada base ──────────────────────────────────────────
# MongoDB
n_usuarios  = mongo_db.usuarios.count_documents({})
n_props     = mongo_db.propiedades.count_documents({'estado': {'$ne': 'eliminada'}})

# PostgreSQL
pg_cur.execute('SELECT COUNT(*) AS n FROM reservas')
n_reservas = pg_cur.fetchone()['n']
pg_cur.execute("SELECT COUNT(*) AS n FROM reservas WHERE estado = 'confirmada'")
n_activas = pg_cur.fetchone()['n']
pg_cur.execute('SELECT COALESCE(SUM(monto),0) AS total FROM pagos WHERE estado = %s', ('completado',))
facturado = pg_cur.fetchone()['total']

# Cassandra
n_resenias = cass_session.execute('SELECT COUNT(*) FROM resenias_by_reserva').one().count

# Neo4j
with neo4j_driver.session() as s:
    n_nodos = s.run('MATCH (n) RETURN count(n) AS n').single()['n']
    n_rels  = s.run('MATCH ()-[r]->() RETURN count(r) AS n').single()['n']

summary = pd.DataFrame([
    {'Base',       'Entidad',              'Cantidad'},
]).from_records([
    {'Base': 'MongoDB',    'Entidad': 'Usuarios',            'Cantidad': n_usuarios},
    {'Base': 'MongoDB',    'Entidad': 'Propiedades activas', 'Cantidad': n_props},
    {'Base': 'PostgreSQL', 'Entidad': 'Reservas (total)',    'Cantidad': n_reservas},
    {'Base': 'PostgreSQL', 'Entidad': 'Reservas confirmadas','Cantidad': n_activas},
    {'Base': 'PostgreSQL', 'Entidad': 'Facturado (completado)', 'Cantidad': f'${int(facturado):,}'},
    {'Base': 'Cassandra',  'Entidad': 'Reseñas',             'Cantidad': n_resenias},
    {'Base': 'Neo4j',      'Entidad': 'Nodos en el grafo',   'Cantidad': n_nodos},
    {'Base': 'Neo4j',      'Entidad': 'Relaciones en el grafo','Cantidad': n_rels},
])
display(summary)

,Base,Entidad,Cantidad
0,MongoDB,Usuarios,12
1,MongoDB,Propiedades activas,6
2,PostgreSQL,Reservas (total),12
3,PostgreSQL,Reservas confirmadas,1
4,PostgreSQL,Facturado (completado),"$2,220,000"
5,Cassandra,Reseñas,8
6,Neo4j,Nodos en el grafo,21
7,Neo4j,Relaciones en el grafo,13


---
## 10. Limpieza: eliminar datos insertados en la sección 6

Borramos exactamente lo que creamos durante el notebook para dejar cada base en el estado original.

| Base | Qué se elimina |
|---|---|
| MongoDB | Usuario, propiedad y cache de reseña creados en sección 6; campo `verificado` del Caso 3 |
| PostgreSQL | Reserva (cascade a pagos) |
| Cassandra | Reseña de las 3 tablas (batch) |
| Neo4j | Nodos y relaciones creados para el usuario y la propiedad del notebook |

In [33]:
# ── MongoDB: eliminar usuario, propiedad y cache de reseña ───────────────────
r1 = mongo_db.usuarios.delete_one({'id': nuevo_guest_id})
r2 = mongo_db.propiedades.delete_one({'id': nueva_prop_id})
r3 = mongo_db.resenias_cache.delete_many({'propiedad_id': nueva_prop_id})

# Revertir campo 'verificado' agregado en el Caso 3
r4 = mongo_db.usuarios.update_many({}, {'$unset': {'verificado': '', 'verificado_en': ''}})

print('MongoDB:')
print(f'  Usuarios eliminados     : {r1.deleted_count}')
print(f'  Propiedades eliminadas  : {r2.deleted_count}')
print(f'  Cache reseñas eliminadas: {r3.deleted_count}')
print(f'  Campo verificado quitado: {r4.modified_count} documentos')

MongoDB:
  Usuarios eliminados     : 1
  Propiedades eliminadas  : 1
  Cache reseñas eliminadas: 1
  Campo verificado quitado: 5 documentos


In [34]:
# ── PostgreSQL: eliminar reserva (cascade a pagos por FK) ────────────────────
try:
    pg_cur.execute('DELETE FROM reservas WHERE id = %s RETURNING id', (reserva_id,))
    deleted = pg_cur.fetchone()
    pg_conn.commit()
    print(f'PostgreSQL:')
    print(f'  Reserva eliminada: {deleted["id"] if deleted else "no encontrada"}')
    print(f'  Pago eliminado por CASCADE (FK ON DELETE CASCADE)')
except Exception as e:
    pg_conn.rollback()
    print(f'Error PostgreSQL: {e}')

PostgreSQL:
  Reserva eliminada: 4dba4c48-1d89-4901-b91d-5e2afeb36749
  Pago eliminado por CASCADE (FK ON DELETE CASCADE)


In [35]:
# ── Cassandra: eliminar reseña de las 3 tablas (batch) ───────────────────────
del_batch = BatchStatement(consistency_level=ConsistencyLevel.ONE)

d1 = cass_session.prepare(
    'DELETE FROM resenias_by_propiedad WHERE propiedad_id = ? AND created_at = ? AND id = ?'
)
d2 = cass_session.prepare(
    'DELETE FROM resenias_by_anfitrion WHERE anfitrion_id = ? AND created_at = ? AND id = ?'
)
d3 = cass_session.prepare(
    'DELETE FROM resenias_by_reserva WHERE reserva_id = ?'
)

del_batch.add(d1, (nueva_prop_id, ts, resenia_id))
del_batch.add(d2, (HOST1, ts, resenia_id))
del_batch.add(d3, (reserva_id,))

cass_session.execute(del_batch)
print('Cassandra:')
print(f'  Reseña {resenia_id[:8]}... eliminada de las 3 tablas (BATCH)')

Cassandra:
  Reseña 5197ddc8... eliminada de las 3 tablas (BATCH)


In [36]:
# ── Neo4j: eliminar nodos y sus relaciones ────────────────────────────────────
with neo4j_driver.session() as s:
    r_u = s.run(
        'MATCH (u:Usuario {id: $id}) DETACH DELETE u RETURN count(u) AS n',
        id=nuevo_guest_id
    ).single()
    r_p = s.run(
        'MATCH (p:Propiedad {id: $id}) DETACH DELETE p RETURN count(p) AS n',
        id=nueva_prop_id
    ).single()

print('Neo4j:')
print(f'  Nodo Usuario eliminado   : {r_u["n"]}')
print(f'  Nodo Propiedad eliminado : {r_p["n"]}')
print(f'  Relaciones DETACH DELETE (ANFITRIO + RESERVO eliminadas automáticamente)')

Neo4j:
  Nodo Usuario eliminado   : 1
  Nodo Propiedad eliminado : 1
  Relaciones DETACH DELETE (ANFITRIO + RESERVO eliminadas automáticamente)


In [37]:
# ── Verificación: los conteos volvieron al estado previo al INSERT ────────────
n_usuarios  = mongo_db.usuarios.count_documents({})
n_props     = mongo_db.propiedades.count_documents({'estado': {'$ne': 'eliminada'}})

pg_cur.execute('SELECT COUNT(*) AS n FROM reservas')
n_reservas = pg_cur.fetchone()['n']

n_resenias = cass_session.execute('SELECT COUNT(*) FROM resenias_by_reserva').one().count

with neo4j_driver.session() as s:
    n_nodos = s.run('MATCH (n) RETURN count(n) AS n').single()['n']
    n_rels  = s.run('MATCH ()-[r]->() RETURN count(r) AS n').single()['n']

    # Confirmar que verificado fue removido
verificado_count = mongo_db.usuarios.count_documents({'verificado': {'$exists': True}})

resumen = pd.DataFrame([
    {'Base': 'MongoDB',    'Entidad': 'Usuarios',              'Cantidad': n_usuarios},
    {'Base': 'MongoDB',    'Entidad': 'Propiedades activas',   'Cantidad': n_props},
    {'Base': 'MongoDB',    'Entidad': 'Campo verificado',      'Cantidad': verificado_count},
    {'Base': 'PostgreSQL', 'Entidad': 'Reservas (total)',      'Cantidad': n_reservas},
    {'Base': 'Cassandra',  'Entidad': 'Reseñas',               'Cantidad': n_resenias},
    {'Base': 'Neo4j',      'Entidad': 'Nodos en el grafo',     'Cantidad': n_nodos},
    {'Base': 'Neo4j',      'Entidad': 'Relaciones en el grafo','Cantidad': n_rels},
])
print('Estado post-limpieza:')
display(resumen)

Estado post-limpieza:


,Base,Entidad,Cantidad
0,MongoDB,Usuarios,11
1,MongoDB,Propiedades activas,5
2,MongoDB,Campo verificado,0
3,PostgreSQL,Reservas (total),11
4,Cassandra,Reseñas,7
5,Neo4j,Nodos en el grafo,19
6,Neo4j,Relaciones en el grafo,11


In [38]:
# ── Cierre de conexiones ──────────────────────────────────────────────────────
pg_cur.close()
pg_conn.close()
cass_session.shutdown()
cass_cluster.shutdown()
neo4j_driver.close()
mongo_client.close()
print('Todas las conexiones cerradas.')

Todas las conexiones cerradas.
